In [27]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import os

In [11]:
#loading the mnist data set
_ = torch.manual_seed(0)

In [12]:
transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,),(0.3081,))])

#load mnist
mnist_trainset=datasets.MNIST(root='./data',train=True, download=True,transform=transform)

#create a dataloader for the training
train_loader=torch.utils.data.DataLoader(mnist_trainset,batch_size=10,shuffle=True)

#now the same for testset
mnist_testset=datasets.MNIST(root='./data',train=False, download=True, transform=transform)
test_loader=torch.utils.data.DataLoader(mnist_testset,batch_size=10,shuffle=True)

#define the device
device='cpu'

In [15]:
#define the model
class verysimplenet(nn.Module):
    def __init__(self,hidden_size_1=100, hidden_size_2=100):
        super(verysimplenet,self).__init__()
        self.linear1=nn.Linear(28*28,hidden_size_1)
        self.linear2=nn.Linear(hidden_size_1,hidden_size_2)
        self.linear3=nn.Linear(hidden_size_2,10)
        self.relu=nn.ReLU()

    def forward(self, img):
        x=img.view(-1,28*28)
        x=self.relu(self.linear1(x))
        x=self.relu(self.linear2(x))
        x=self.linear3(x)
        return x

In [16]:
net=verysimplenet().to(device)

In [17]:
#train the model
def train(train_loader,net,epochs=5,total_iterations_limit=None):
    cross_el=nn.CrossEntropyLoss()
    optimizer=torch.optim.Adam(net.parameters(),lr=0.001)

    total_iterations=0

    for epoch in range(epochs):
        net.train()

        loss_sum=0
        num_iterations=0

        data_iterator=tqdm(train_loader,desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total=total_iterations_limit

        for data in data_iterator:
            num_iterations+=1
            total_iterations+=1
            x,y =data
            x=x.to(device)
            y=y.to(device)
            optimizer.zero_grad()
            output=net(x.view(-1, 28*28))
            loss=cross_el(output,y)
            loss_sum+=loss.item()
            avg_loss=loss_sum/num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >=total_iterations_limit:
                return

def print_size_of_model(model):
    torch.save(model.state_dict(), "temp_delme.p")
    print("Size (KB): ",os.path.getsize("temp_delme.p")/1e3)
    os.remove("temp_delme.p")

MODEL_FILENAME = "simplenet_ptq.pt"

if Path(MODEL_FILENAME).exists():
    net.load_state_dict(torch.load(MODEL_FILENAME))
    print("Loaded model from disk")
else:
    train(train_loader,net,epochs=1)
    torch.save(net.state_dict(),MODEL_FILENAME)


Epoch 1: 100%|██████████| 6000/6000 [03:39<00:00, 27.29it/s, loss=0.223]


In [18]:
#testing loop
def test(model: nn.Module, total_iterations: int= None):
    correct=0
    total=0

    iterations=0

    model.eval()

    with torch.no_grad():
        for data in tqdm(test_loader,desc="Testing"):
            x,y=data
            x=x.to(device)
            y=y.to(device)
            output=model(x.view(-1,784))
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct+=1
                total+=1
            iterations+=1
            if total_iterations is not None and iterations >= total_iterations:
                break
    print(f"Accuracy: {round(correct/total,3)}")


In [23]:
#weight and size of model before quantization
#weight matrix
print("Weights before quantization: ")
print(net.linear1.weight)
print(net.linear1.weight.dtype)
#size
print("Size of Model before quantization:")
print_size_of_model(net)
#Accuracy
print("Accuracy of Model before quantization: ")
test(net)

Weights before quantization: 
Parameter containing:
tensor([[-0.0127,  0.0067, -0.0419,  ...,  0.0095, -0.0087, -0.0104],
        [-0.0197, -0.0149, -0.0104,  ..., -0.0202, -0.0059, -0.0299],
        [ 0.0199,  0.0550,  0.0068,  ...,  0.0197,  0.0413,  0.0481],
        ...,
        [ 0.0278,  0.0316, -0.0031,  ..., -0.0084,  0.0108, -0.0261],
        [ 0.0056,  0.0137,  0.0458,  ...,  0.0261,  0.0261,  0.0256],
        [ 0.0102,  0.0049, -0.0093,  ...,  0.0271, -0.0221, -0.0020]],
       requires_grad=True)
torch.float32
Size of Model before quantization:
Size (KB):  361.401
Accuracy of Model before quantization: 


Testing: 100%|██████████| 1000/1000 [00:07<00:00, 139.24it/s]

Accuracy: 0.961


In [24]:
#insert min-max observers in the model
#define the model
class quantizedverysimplenet(nn.Module):
    def __init__(self,hidden_size_1=100, hidden_size_2=100):
        super(quantizedverysimplenet,self).__init__()
        self.quant=torch.quantization.QuantStub()#these stubs are used by pytorch to do quantization on the fly
        self.linear1=nn.Linear(28*28,hidden_size_1)
        self.linear2=nn.Linear(hidden_size_1,hidden_size_2)
        self.linear3=nn.Linear(hidden_size_2,10)
        self.relu=nn.ReLU()
        self.dequant= torch.quantization.DeQuantStub()

    def forward(self, img):
        x=img.view(-1,28*28)
        x=self.quant(x)
        x=self.relu(self.linear1(x))
        x=self.relu(self.linear2(x))
        x=self.linear3(x)
        x=self.dequant(x)
        return x
        


In [25]:
net_quantized =quantizedverysimplenet().to(device)
#copy weights from unquantized model
net_quantized.load_state_dict(net.state_dict())
net_quantized.eval()

net_quantized.qconfig=torch.ao.quantization.default_qconfig
net_quantized=torch.ao.quantization.prepare(net_quantized)#insert observers
net_quantized

/tmp/ipykernel_170093/2101534010.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  net_quantized=torch.ao.quantization.prepare(net_quantized)#insert observers


quantizedverysimplenet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)

In [26]:
#calibrate the model using the test set
test(net_quantized)

Testing: 100%|██████████| 1000/1000 [00:03<00:00, 280.31it/s]

Accuracy: 0.961
